In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from scipy.optimize import least_squares

from inverse_pinn_model import InversePINN

In [ ]:
# Experiment settings
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
dtype = torch.float64

TRAIN_POOL_SIZE = 200

data_sizes = [
    2,
    30,
    50,
    100,
    120,
    150,
    200,
]

N_f = 2000
N_terminal = 500
N_boundary = 500

epsilon = 1e-4

hidden_width = 64
hidden_layers = 4

r_initial = 0.10
sigma_initial = 0.40

learning_rate = 1e-3
num_epochs = 30000

print_every = 5000
history_every = 100

# Notebook 13.1: repeat each PINN calibration 10 times
PINN_RUNS = 10

# Real-data lambda setting selected from Notebook 11.1
lambda_data = 1.0
lambda_pde = 1.0
lambda_terminal = 0.01
lambda_boundary = 0.001

print("device =", device)

if torch.cuda.is_available():
    print(
        "GPU =",
        torch.cuda.get_device_name(0)
    )

print("data_sizes =", data_sizes)
print()
print("N_f =", N_f)
print("N_terminal =", N_terminal)
print("N_boundary =", N_boundary)
print("PINN_RUNS =", PINN_RUNS)
print()
print("lambdas =", {
    "data": lambda_data,
    "pde": lambda_pde,
    "terminal": lambda_terminal,
    "boundary": lambda_boundary,
})


In [ ]:

# ------------------------------------------------------------
# Automatically locate NDX_call_data.csv
# ------------------------------------------------------------

from pathlib import Path

_filename = "NDX_call_data.csv"

_search_roots = [
    Path.cwd().resolve()
]

try:
    _notebook_dir = Path(__file__).resolve().parent
    _search_roots.append(_notebook_dir)
except NameError:
    pass


_candidates = []

for _root in _search_roots:

    _candidates.extend([
        _root / _filename,
        _root / "data" / _filename,
        _root.parent / _filename,
        _root.parent / "data" / _filename,
    ])


data_path = None


# First check common locations
for _candidate in _candidates:

    if _candidate.is_file():

        data_path = _candidate.resolve()
        break


# Then recursively search the current project
if data_path is None:

    for _root in _search_roots:

        try:

            _matches = list(
                _root.rglob(
                    _filename
                )
            )

        except Exception:

            _matches = []


        if _matches:

            data_path = (
                _matches[0]
                .resolve()
            )

            break


if data_path is None:

    raise FileNotFoundError(
        "\nCould not find NDX_call_data.csv.\n"
        "Make sure the CSV is included somewhere inside the downloaded project folder."
    )


print(
    "Using data file:",
    data_path
)


raw_df = pd.read_csv(
    data_path
)

unnamed_columns = [
    col
    for col in raw_df.columns
    if str(col).startswith("Unnamed:")
]

if unnamed_columns:
    raw_df = raw_df.drop(
        columns=unnamed_columns
    )


raw_df = raw_df.reset_index(
    drop=True
)


# Save original row identity
raw_df["original_row"] = np.arange(
    len(raw_df)
)


# If market_price is not already: use bid/ask midpoint
if "market_price" not in raw_df.columns:

    if not {
        "best_bid",
        "best_offer"
    }.issubset(
        raw_df.columns
    ):

        raise KeyError(
            "Need market_price or "
            "best_bid and best_offer."
        )

    raw_df["market_price"] = (
        raw_df["best_bid"]
        + raw_df["best_offer"]
    ) / 2.0

# Strike Price

K = float(
    raw_df[
        "strike_price"
    ]
    .dropna()
    .iloc[0]
)

# Keep required columns 
real_df = raw_df[
    [
        "original_row",
        "current_time",
        "spot_price",
        "market_price"
    ]
].copy()

# remove invalid observations
real_df = (
    real_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
)


real_df = real_df[
    (
        real_df["current_time"]
        >= 0.0
    )
    &
    (
        real_df["spot_price"]
        > 0.0
    )
    &
    (
        real_df["market_price"]
        >= 0.0
    )
].copy()


real_df = real_df.reset_index(
    drop=True
)


print(
    "Number of usable observations =",
    len(real_df)
)

print(
    "K =",
    K
)

In [ ]:
T = float(
    real_df[
        "current_time"
    ].max()
)


real_df["t"] = (
    T
    - real_df["current_time"]
)


# Physical stock price
real_df["S"] = (
    real_df["spot_price"]
)


# Physical option price
real_df["V"] = (
    real_df["market_price"]
)


print("T =", T)

print()

print(
    "t min =",
    real_df["t"].min()
)

print(
    "t max =",
    real_df["t"].max()
)

print()

print(
    "S min =",
    real_df["S"].min()
)

print(
    "S max =",
    real_df["S"].max()
)

print()

print(
    "V min =",
    real_df["V"].min()
)

print(
    "V max =",
    real_df["V"].max()
)


real_df.head()

In [ ]:
# Define physical time / left-price domain
t_min = 0.0
t_max = T

S_min = 0.0


In [ ]:
# Convert real observations to tensor
t_all = torch.tensor(
    real_df[
        "t"
    ].to_numpy(),
    dtype=dtype,
    device=device
).reshape(
    -1,
    1
)


S_all = torch.tensor(
    real_df[
        "S"
    ].to_numpy(),
    dtype=dtype,
    device=device
).reshape(
    -1,
    1
)


V_all = torch.tensor(
    real_df[
        "V"
    ].to_numpy(),
    dtype=dtype,
    device=device
).reshape(
    -1,
    1
)

# u_all for PINN and V_all for regression
u_all = (
    V_all / K
)


N_obs = len(
    real_df
)


print(
    "N observations =",
    N_obs
)

print()

print(
    "t_all shape =",
    t_all.shape
)

print(
    "S_all shape =",
    S_all.shape
)

print(
    "V_all shape =",
    V_all.shape
)

print(
    "u_all shape =",
    u_all.shape
)

### Random train/test split


In [ ]:
# Random train/test split
#
# A fresh random permutation is created each time this cell is run.
# The first 200 observations form the shared training pool and the
# remaining observations form the test set for this run.
random_order_np = np.random.permutation(
    N_obs
)

train_pool_idx_np = random_order_np[
    :TRAIN_POOL_SIZE
]

test_idx_np = random_order_np[
    TRAIN_POOL_SIZE:
]

# Convert to torch indices
train_pool_idx = torch.tensor(
    train_pool_idx_np,
    dtype=torch.long,
    device=device
)

test_idx = torch.tensor(
    test_idx_np,
    dtype=torch.long,
    device=device
)

# Nested training subsets, exactly as in Notebook 11:
# D_2 subset D_30 subset ... subset D_200.
sample_indices = {
    n_data: train_pool_idx[:n_data]
    for n_data in data_sizes
}

# Same test set for every n_data and both methods within this run
t_test = t_all[test_idx]
S_test = S_all[test_idx]
V_test = V_all[test_idx]
u_test = u_all[test_idx]

# Keep the Notebook 11 spatial-domain definition.
# Only the collocation sampling scheme changes here.
S_data_max = float(
    real_df["S"].max()
)

S_max = (
    10.0
    * S_data_max
)

print("Random split created")
print("Training pool size =", len(train_pool_idx))
print("Test set size =", len(test_idx))
print("S_data_max =", S_data_max)
print("S_max =", S_max)


In [ ]:
# Helper: return the exact same observed subset for Regression and PINN
def get_training_subset(n_data):

    selected_idx = sample_indices[n_data]

    t_train = t_all[selected_idx]
    S_train = S_all[selected_idx]
    V_train = V_all[selected_idx]
    u_train = u_all[selected_idx]

    return (
        selected_idx,
        t_train,
        S_train,
        V_train,
        u_train,
    )


In [ ]:
# Safety checks

# n_data = 200 is exactly the full 200-point training pool
assert torch.equal(
    sample_indices[200],
    train_pool_idx
)

# Check nested datasets
for n_small, n_large in zip(
    data_sizes[:-1],
    data_sizes[1:],
):

    small_set = set(
        sample_indices[n_small]
        .detach()
        .cpu()
        .numpy()
        .tolist()
    )

    large_set = set(
        sample_indices[n_large]
        .detach()
        .cpu()
        .numpy()
        .tolist()
    )

    assert small_set.issubset(
        large_set
    )

# Train and test must not overlap
assert set(train_pool_idx_np).isdisjoint(
    set(test_idx_np)
)

# Together they must cover every usable observation exactly once
assert set(train_pool_idx_np).union(
    set(test_idx_np)
) == set(range(N_obs))

print("Total observations =", N_obs)
print("Training pool =", len(train_pool_idx))
print("Test set =", len(test_idx))
print("Random train/test separation = OK")
print("Nested n_data subsets = OK")
print("Regression/PINN shared observed subsets = OK")


In [ ]:
# Save the exact random train/test split used in this run
split_df = real_df[
    [
        "original_row",
        "current_time",
        "t",
        "S",
        "V"
    ]
].copy()

split_df[
    "split"
] = "test"

split_df.loc[
    train_pool_idx_np,
    "split"
] = "train_pool"

# Position in the fresh random permutation used for this run
random_rank = np.empty(
    N_obs,
    dtype=int
)

random_rank[
    random_order_np
] = np.arange(
    1,
    N_obs + 1
)

split_df[
    "random_rank"
] = random_rank

# Position inside the 200-point training pool
train_order = np.full(
    N_obs,
    np.nan
)

for order, idx in enumerate(
    train_pool_idx_np,
    start=1
):

    train_order[
        idx
    ] = order

split_df[
    "train_pool_order"
] = train_order

split_df.to_csv(
    "notebook13_1_random_mixed_train_test_split.csv",
    index=False
)


### Black-Scholes Eq.(5)

In [ ]:
def black_scholes_call(
    t,
    S,
    K,
    T,
    r,
    sigma
):

    tau = (
        T - t
    )


    eps = 1e-12


    tau_safe = torch.clamp(
        tau,
        min=eps
    )


    S_safe = torch.clamp(
        S,
        min=eps
    )

# d1
    d1 = (

        torch.log(
            S_safe / K
        )

        + (
            r
            + 0.5
            * sigma ** 2
        )
        * tau_safe

    ) / (

        sigma
        * torch.sqrt(
            tau_safe
        )
    )

# d2
    d2 = (

        d1

        - sigma
        * torch.sqrt(
            tau_safe
        )
    )


    # Normal CDF
    N_d1 = (

        0.5

        * (
            1.0

            + torch.erf(
                d1
                / np.sqrt(2.0)
            )
        )
    )


    N_d2 = (

        0.5

        * (
            1.0

            + torch.erf(
                d2
                / np.sqrt(2.0)
            )
        )
    )

  # BS call price
    V = (

        S_safe
        * N_d1

        - K
        * torch.exp(
            -r
            * tau_safe
        )
        * N_d2
    )


    # Exact terminal payoff
    payoff = torch.relu(
        S - K
    )


    V = torch.where(
        tau <= eps,
        payoff,
        V
    )
    # payoff =  max(S-K, 0)

    V = torch.where(
        S <= eps,
        torch.zeros_like(V),
        V
    )


    return V


### Regression calibration for each n_data


In [ ]:
def fit_regression_for_n_data(
    n_data
):

    (
        selected_idx,
        t_reg_train,
        S_reg_train,
        V_reg_train,
        _
    ) = get_training_subset(
        n_data
    )

    # residual = BS price - real market price
    def regression_residuals(
        parameters
    ):

        r_value = float(
            parameters[0]
        )

        sigma_value = float(
            parameters[1]
        )

        with torch.no_grad():

            V_prediction = black_scholes_call(
                t=t_reg_train,
                S=S_reg_train,
                K=K,
                T=T,
                r=r_value,
                sigma=sigma_value
            )

        residuals = (
            V_prediction
            - V_reg_train
        )

        return (
            residuals
            .cpu()
            .numpy()
            .reshape(-1)
        )

    # Nonlinear least-squares calibration
    regression_fit = least_squares(
        regression_residuals,
        x0=np.array(
            [
                r_initial,
                sigma_initial
            ],
            dtype=float
        ),
        bounds=(
            np.array(
                [
                    1e-8,
                    1e-8
                ]
            ),
            np.array(
                [
                    np.inf,
                    np.inf
                ]
            )
        ),
        max_nfev=10000
    )

    r_tilde = float(
        regression_fit.x[0]
    )

    sigma_tilde = float(
        regression_fit.x[1]
    )

    with torch.no_grad():

        # Training prediction for this n_data subset
        V_reg_train_prediction = black_scholes_call(
            t=t_reg_train,
            S=S_reg_train,
            K=K,
            T=T,
            r=r_tilde,
            sigma=sigma_tilde
        )

        regression_train_mse = torch.mean(
            (
                V_reg_train_prediction
                - V_reg_train
            ) ** 2
        ).item()

        # Same fixed test set for every n_data
        V_regression_test = black_scholes_call(
            t=t_test,
            S=S_test,
            K=K,
            T=T,
            r=r_tilde,
            sigma=sigma_tilde
        )

        mse_1_regression = torch.mean(
            (
                V_regression_test
                - V_test
            ) ** 2
        ).item()

    result = {
        "n_data": n_data,
        "r_tilde": r_tilde,
        "sigma_tilde": sigma_tilde,
        "regression_train_mse": regression_train_mse,
        "mse_1_regression_eq5_test": mse_1_regression,
        "regression_success": bool(regression_fit.success),
    }

    print()
    print("-" * 70)
    print(f"Regression with n_data = {n_data}")
    print(f"r_tilde = {r_tilde:.10f}")
    print(f"sigma_tilde = {sigma_tilde:.10f}")
    print(f"Regression train MSE = {regression_train_mse:.10f}")
    print(f"Regression test MSE = {mse_1_regression:.10f}")
    print("Optimization success =", regression_fit.success)

    return (
        result,
        V_regression_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


### MSE #1: Regression


In [ ]:
# BS PDE residual

def black_scholes_residual(
    model,
    t,
    S
):

    t_input = (
        t.clone()
        .detach()
        .requires_grad_(True)
    )


    S_input = (
        S.clone()
        .detach()
        .requires_grad_(True)
    )


    u = model(
        t_input,
        S_input
    )


    # du/dt
    u_t = torch.autograd.grad(

        outputs=u,
        inputs=t_input,

        grad_outputs=torch.ones_like(
            u
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    # du/dS
    u_S = torch.autograd.grad(

        outputs=u,
        inputs=S_input,

        grad_outputs=torch.ones_like(
            u
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    # d2u/dS2
    u_SS = torch.autograd.grad(

        outputs=u_S,
        inputs=S_input,

        grad_outputs=torch.ones_like(
            u_S
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    residual = (

        u_t

        + 0.5
        * model.sigma ** 2
        * S_input ** 2
        * u_SS

        + model.r
        * S_input
        * u_S

        - model.r
        * u
    )


    return residual


### PINN PDE residual + loss

In [ ]:
# PINN loss
def compute_losses(

    model,

    t_train,
    S_train,
    u_train,

    t_f,
    S_f,

    t_terminal,
    S_terminal,

    t_boundary,
    S_boundary_left,
    S_boundary_right
):

    # 1. Data loss

    u_data_prediction = model(
        t_train,
        S_train
    )


    loss_data = torch.mean(

        (
            u_data_prediction
            - u_train
        ) ** 2
    )


    # 2. PDE loss

    pde_prediction = (
        black_scholes_residual(
            model,
            t_f,
            S_f
        )
    )


    loss_pde = torch.mean(
        pde_prediction ** 2
    )


    # 3. Terminal loss
    #
    # V(T,S) = max(S-K,0)
    #
    # u(T,S) = V/K

    u_terminal_prediction = model(
        t_terminal,
        S_terminal
    )


    u_terminal_true = (

        torch.relu(
            S_terminal
            - K
        )

        / K
    )


    loss_terminal = torch.mean(

        (
            u_terminal_prediction
            - u_terminal_true
        ) ** 2
    )

    # 4. Left boundary
    #
    # V(t,0) = 0

    u_left_prediction = model(
        t_boundary,
        S_boundary_left
    )


    u_left_true = (
        torch.zeros_like(
            u_left_prediction
        )
    )


    loss_boundary_left = torch.mean(

        (
            u_left_prediction
            - u_left_true
        ) ** 2
    )


    # 5. Right boundary
    #
    # V approximately
    #
    # S - K exp[-r(T-t)]

    u_right_prediction = model(
        t_boundary,
        S_boundary_right
    )


    u_right_true = (

        S_boundary_right

        - K
        * torch.exp(

            -model.r

            * (
                T
                - t_boundary
            )
        )

    ) / K


    loss_boundary_right = torch.mean(

        (
            u_right_prediction
            - u_right_true
        ) ** 2
    )


    loss_boundary = (

        loss_boundary_left
        + loss_boundary_right
    )


    # Total loss

    total_loss = (

        lambda_data
        * loss_data

        + lambda_pde
        * loss_pde

        + lambda_terminal
        * loss_terminal

        + lambda_boundary
        * loss_boundary
    )


    return {

        "total":
            total_loss,

        "data":
            loss_data,

        "pde":
            loss_pde,

        "terminal":
            loss_terminal,

        "boundary":
            loss_boundary
    }

In [ ]:
# Shared PINN physics points for all n_data in this run
#
# These points are generated once when this cell runs and then reused
# across all n_data values in the same notebook run.

def rand_cpu(
    n_rows
):
    return torch.rand(
        n_rows,
        1,
        dtype=dtype,
        device="cpu"
    )


# PDE collocation points = 1000 global + 1000 local around K
N_f_global = (
    N_f // 2
)

N_f_local = (
    N_f
    - N_f_global
)

S_local_min = max(
    S_min + epsilon,
    0.8 * K
)

S_local_max = min(
    S_max - epsilon,
    1.2 * K
)

# t covers the full PDE time domain
t_f_cpu = (
    t_min
    + epsilon
    + (
        t_max
        - t_min
        - 2.0 * epsilon
    )
    * rand_cpu(
        N_f
    )
)

# Global S points cover the original large domain
S_f_global_cpu = (
    S_min
    + epsilon
    + (
        S_max
        - S_min
        - 2.0 * epsilon
    )
    * rand_cpu(
        N_f_global
    )
)

# Local S points increase PDE density near the strike/data region
S_f_local_cpu = (
    S_local_min
    + (
        S_local_max
        - S_local_min
    )
    * rand_cpu(
        N_f_local
    )
)

S_f_cpu = torch.cat(
    [
        S_f_global_cpu,
        S_f_local_cpu
    ],
    dim=0
)

t_f = t_f_cpu.to(
    device
)

S_f = S_f_cpu.to(
    device
)

# Terminal-condition points
t_terminal = torch.full(
    (N_terminal, 1),
    fill_value=T,
    dtype=dtype,
    device=device
)

S_terminal_cpu = (
    S_min
    + (
        S_max
        - S_min
    )
    * rand_cpu(
        N_terminal
    )
)

S_terminal = S_terminal_cpu.to(
    device
)

# Boundary-condition points
t_boundary_cpu = (
    t_min
    + (
        t_max
        - t_min
    )
    * rand_cpu(
        N_boundary
    )
)

t_boundary = t_boundary_cpu.to(
    device
)

S_boundary_left = torch.full(
    (N_boundary, 1),
    fill_value=S_min,
    dtype=dtype,
    device=device
)

S_boundary_right = torch.full(
    (N_boundary, 1),
    fill_value=S_max,
    dtype=dtype,
    device=device
)

print("Shared physics points created")
print()
print("PDE points =", N_f)
print("Global PDE points =", N_f_global)
print("Local PDE points =", N_f_local)
print(
    "Local S interval =",
    S_local_min,
    "to",
    S_local_max
)
print()
print(
    "PDE t range:",
    float(t_f.min()),
    "to",
    float(t_f.max())
)
print(
    "Global PDE S range:",
    float(S_f_global_cpu.min()),
    "to",
    float(S_f_global_cpu.max())
)
print(
    "Local PDE S range:",
    float(S_f_local_cpu.min()),
    "to",
    float(S_f_local_cpu.max())
)
print(
    "Terminal S range:",
    float(S_terminal.min()),
    "to",
    float(S_terminal.max())
)
print(
    "Right boundary S =",
    float(S_boundary_right[0])
)


In [ ]:
# Create TEN random model initializations for this notebook run.
# This preserves Notebook 13's comparison logic:
# - initialization #1 is reused across every n_data,
# - initialization #2 is reused across every n_data,
# - initialization #10 is reused across every n_data.
#
# Therefore each n_data has 10 PINN runs, while run-to-run initialization
# differs and the corresponding run number remains comparable across n_data.

base_state_dicts = []

for run_number in range(1, PINN_RUNS + 1):

    base_model = InversePINN(
        t_min=t_min,
        t_max=t_max,
        S_min=S_min,
        S_max=S_max,
        hidden_width=hidden_width,
        hidden_layers=hidden_layers,
        r_initial=r_initial,
        sigma_initial=sigma_initial
    ).to(
        device=device,
        dtype=dtype
    )

    base_state_dicts.append(
        copy.deepcopy(
            base_model.state_dict()
        )
    )

    print(
        f"Initialization {run_number:2d} | "
        f"Initial r = {base_model.r.item():.10f} | "
        f"Initial sigma = {base_model.sigma.item():.10f}"
    )

    del base_model

print()
print(
    f"Created {PINN_RUNS} unseeded random model initializations."
)


In [ ]:
# Save the exact random realization used by Notebook 13.1
#
# The random split and shared physics points are exactly the same objects
# used by every n_data and all 10 PINN repetitions in this notebook run.
torch.save(
    {
        "split_type": "random",
        "random_order_indices": torch.tensor(
            random_order_np,
            dtype=torch.long
        ),
        "train_pool_indices": train_pool_idx.detach().cpu(),
        "test_indices": test_idx.detach().cpu(),
        "sample_indices": {
            int(n): sample_indices[n].detach().cpu()
            for n in data_sizes
        },
        "collocation_scheme": "1000_global_1000_local_around_K",
        "N_f_global": N_f_global,
        "N_f_local": N_f_local,
        "S_local_min": S_local_min,
        "S_local_max": S_local_max,
        "t_f": t_f.detach().cpu(),
        "S_f": S_f.detach().cpu(),
        "t_terminal": t_terminal.detach().cpu(),
        "S_terminal": S_terminal.detach().cpu(),
        "t_boundary": t_boundary.detach().cpu(),
        "S_boundary_left": S_boundary_left.detach().cpu(),
        "S_boundary_right": S_boundary_right.detach().cpu(),
        "K": K,
        "T": T,
        "S_max": S_max,
        "lambda_data": lambda_data,
        "lambda_pde": lambda_pde,
        "lambda_terminal": lambda_terminal,
        "lambda_boundary": lambda_boundary,
        "pinn_runs": PINN_RUNS,
        "base_state_dicts": [
            {
                k: v.detach().cpu()
                for k, v in state_dict.items()
            }
            for state_dict in base_state_dicts
        ],
    },
    "notebook13_1_random_mixed_collocation_realization.pt"
)

print(
    "Saved: "
    "notebook13_1_random_mixed_collocation_realization.pt"
)


In [ ]:
# Train ONE PINN for a given n_data
def train_pinn_for_n_data(
    n_data,
    run_number,
    initial_state_dict
):

    start_time = time.time()


    # Use exactly the same observed subset as Regression
    (
        selected_idx,
        t_train,
        S_train,
        V_train,
        u_train
    ) = get_training_subset(
        n_data
    )


    # Fresh model object for this n_data / repetition.
    # Load the corresponding unseeded initialization for this repetition.
    model = InversePINN(

        t_min=t_min,
        t_max=t_max,

        S_min=S_min,
        S_max=S_max,

        hidden_width=hidden_width,
        hidden_layers=hidden_layers,

        r_initial=r_initial,
        sigma_initial=sigma_initial

    ).to(
        device=device,
        dtype=dtype
    )

    model.load_state_dict(
        copy.deepcopy(
            initial_state_dict
        )
    )


    # Optimizer

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )


    # History

    history = {

        "epoch": [],

        "total_loss": [],
        "data_loss": [],
        "pde_loss": [],
        "terminal_loss": [],
        "boundary_loss": [],

        "r_estimate": [],
        "sigma_estimate": []
    }


    print()
    print(
        "=" * 70
    )

    print(
        f"Training PINN with n_data = {n_data}, "
        f"run = {run_number}/{PINN_RUNS}"
    )

    print(
        "=" * 70
    )


    # Training

    for epoch in range(
        1,
        num_epochs + 1
    ):

        model.train()


        optimizer.zero_grad(
            set_to_none=True
        )


        losses = compute_losses(

            model=model,

            t_train=t_train,
            S_train=S_train,
            u_train=u_train,

            t_f=t_f,
            S_f=S_f,

            t_terminal=t_terminal,
            S_terminal=S_terminal,

            t_boundary=t_boundary,
            S_boundary_left=S_boundary_left,
            S_boundary_right=S_boundary_right
        )


        losses[
            "total"
        ].backward()


        optimizer.step()


        # Save history

        if (
            epoch == 1

            or epoch
            % history_every
            == 0

            or epoch
            == num_epochs
        ):

            history[
                "epoch"
            ].append(
                epoch
            )


            history[
                "total_loss"
            ].append(
                losses[
                    "total"
                ].item()
            )


            history[
                "data_loss"
            ].append(
                losses[
                    "data"
                ].item()
            )


            history[
                "pde_loss"
            ].append(
                losses[
                    "pde"
                ].item()
            )


            history[
                "terminal_loss"
            ].append(
                losses[
                    "terminal"
                ].item()
            )


            history[
                "boundary_loss"
            ].append(
                losses[
                    "boundary"
                ].item()
            )


            history[
                "r_estimate"
            ].append(
                model.r.item()
            )


            history[
                "sigma_estimate"
            ].append(
                model.sigma.item()
            )


        # Print progress
        if (
            epoch == 1

            or epoch
            % print_every
            == 0

            or epoch
            == num_epochs
        ):

            print(

                f"Epoch {epoch:5d} | "

                f"Total: "
                f"{losses['total'].item():.6e} | "

                f"Data: "
                f"{losses['data'].item():.6e} | "

                f"PDE: "
                f"{losses['pde'].item():.6e} | "

                f"r: "
                f"{model.r.item():.8f} | "

                f"sigma: "
                f"{model.sigma.item():.8f}"
            )


    # Final PINN parameters

    r_pinn = (
        model.r.item()
    )


    sigma_pinn = (
        model.sigma.item()
    )

    # TEST SET

    model.eval()


    with torch.no_grad():

        # MSE #2
        #
        # NN itself predicts u = V/K
        # Convert back to physical V
        u_pinn_test = model(
            t_test,
            S_test
        )


        V_pinn_nn_test = (
            K
            * u_pinn_test
        )


        mse_2_pinn_nn = (
            torch.mean(

                (
                    V_pinn_nn_test
                    - V_test
                ) ** 2

            )
            .item()
        )


        # MSE #3
        #
        # Take r_PINN and sigma_PINN
        # and plug them into Eq.(5)

        V_pinn_eq5_test = (
            black_scholes_call(

                t=t_test,
                S=S_test,

                K=K,
                T=T,

                r=r_pinn,
                sigma=sigma_pinn
            )
        )


        mse_3_pinn_eq5 = (
            torch.mean(

                (
                    V_pinn_eq5_test
                    - V_test
                ) ** 2

            )
            .item()
        )


    runtime_minutes = (

        time.time()
        - start_time

    ) / 60.0


    # Result dictionary

    result = {

        "n_data":
            n_data,

        "run":
            run_number,

        "r_pinn":
            r_pinn,

        "sigma_pinn":
            sigma_pinn,

        "mse_2_pinn_nn_test":
            mse_2_pinn_nn,

        "mse_3_pinn_eq5_test":
            mse_3_pinn_eq5,

        "runtime_minutes":
            runtime_minutes
    }


    print()

    print(
        f"n_data = {n_data}, run = {run_number} completed"
    )

    print()

    print(
        f"r_PINN = "
        f"{r_pinn:.10f}"
    )

    print(
        f"sigma_PINN = "
        f"{sigma_pinn:.10f}"
    )

    print()

    print(
        "MSE #2 PINN NN =",
        mse_2_pinn_nn
    )

    print(
        "MSE #3 PINN r,sigma -> Eq.(5) =",
        mse_3_pinn_eq5
    )

    print()

    print(
        f"Runtime = "
        f"{runtime_minutes:.2f} minutes"
    )

    print(
        "-" * 70
    )


    return (

        result,

        model,

        history,

        V_pinn_nn_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1),

        V_pinn_eq5_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


In [ ]:
# Run Regression once and PINN 10 times for every n_data
#
# Regression logic is unchanged from Notebook 13.


results = []
all_pinn_run_results = []

# Fixed test-set rows
test_predictions_df = (
    real_df
    .iloc[test_idx_np][
        [
            "original_row",
            "current_time",
            "t",
            "S",
            "V"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

V_test_np = (
    V_test
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

for n_data in data_sizes:

    print()
    print("#" * 78)
    print(
        f"n_data = {n_data}: Regression once + "
        f"{PINN_RUNS} PINN repetitions"
    )
    print("#" * 78)

    (
        regression_result_n,
        V_regression_test_n
    ) = fit_regression_for_n_data(
        n_data
    )

    pinn_run_results_n = []
    pinn_nn_predictions_n = []
    histories_n = []

    for run_number in range(
        1,
        PINN_RUNS + 1
    ):

        (
            pinn_result_run,
            model_run,
            history_run,
            V_pinn_nn_test_run,
            V_pinn_eq5_test_run
        ) = train_pinn_for_n_data(
            n_data=n_data,
            run_number=run_number,
            initial_state_dict=base_state_dicts[
                run_number - 1
            ]
        )

        pinn_run_results_n.append(
            pinn_result_run
        )

        all_pinn_run_results.append(
            pinn_result_run
        )

        pinn_nn_predictions_n.append(
            V_pinn_nn_test_run
        )

        history_run_df = pd.DataFrame(
            history_run
        )

        history_run_df.insert(
            0,
            "run",
            run_number
        )

        histories_n.append(
            history_run_df
        )

        # Save every individual training history
        history_run_df.to_csv(
            (
                f"notebook13_1_random_mixed_"
                f"history_n{n_data}_run{run_number}.csv"
            ),
            index=False
        )

        # Save every individual PINN checkpoint
        torch.save(
            {
                "n_data": n_data,
                "run": run_number,
                "model_state_dict": model_run.state_dict(),
                "selected_indices": sample_indices[n_data].cpu(),
                "train_pool_indices": train_pool_idx.cpu(),
                "test_indices": test_idx.cpu(),
                "regression_result": regression_result_n,
                "pinn_result": pinn_result_run,
                "K": K,
                "T": T,
                "t_min": t_min,
                "t_max": t_max,
                "S_min": S_min,
                "S_max": S_max,
                "S_data_max": S_data_max,
                "N_f": N_f,
                "N_terminal": N_terminal,
                "N_boundary": N_boundary,
                "hidden_width": hidden_width,
                "hidden_layers": hidden_layers,
                "learning_rate": learning_rate,
                "num_epochs": num_epochs,
                "lambda_data": lambda_data,
                "lambda_pde": lambda_pde,
                "lambda_terminal": lambda_terminal,
                "lambda_boundary": lambda_boundary,
                "split_type": "random",
                "collocation_scheme": "1000_global_1000_local_around_K",
                "N_f_global": N_f_global,
                "N_f_local": N_f_local,
                "S_local_min": S_local_min,
                "S_local_max": S_local_max,
                "pinn_runs": PINN_RUNS,
            },
            (
                f"notebook13_1_random_mixed_"
                f"pinn_n{n_data}_run{run_number}.pt"
            )
        )

        del model_run

    r_runs = np.array(
        [
            run_result["r_pinn"]
            for run_result in pinn_run_results_n
        ],
        dtype=float
    )

    sigma_runs = np.array(
        [
            run_result["sigma_pinn"]
            for run_result in pinn_run_results_n
        ],
        dtype=float
    )

    runtime_runs = np.array(
        [
            run_result["runtime_minutes"]
            for run_result in pinn_run_results_n
        ],
        dtype=float
    )

    mse2_runs = np.array(
        [
            run_result["mse_2_pinn_nn_test"]
            for run_result in pinn_run_results_n
        ],
        dtype=float
    )

    mse3_runs = np.array(
        [
            run_result["mse_3_pinn_eq5_test"]
            for run_result in pinn_run_results_n
        ],
        dtype=float
    )

    r_pinn_mean = float(
        np.mean(
            r_runs
        )
    )

    sigma_pinn_mean = float(
        np.mean(
            sigma_runs
        )
    )

    r_pinn_std = float(
        np.std(
            r_runs,
            ddof=1
        )
    )

    sigma_pinn_std = float(
        np.std(
            sigma_runs,
            ddof=1
        )
    )

    # Average the 10 NN price predictions FIRST
    V_pinn_nn_test_mean = np.mean(
        np.stack(
            pinn_nn_predictions_n,
            axis=0
        ),
        axis=0
    )

    # Then compute the usual test MSE #2
    mse_2_pinn_nn = float(
        np.mean(
            (
                V_pinn_nn_test_mean
                - V_test_np
            ) ** 2
        )
    )

    # Use averaged r and sigma in Eq.(5), then compute usual test MSE #3
    with torch.no_grad():

        V_pinn_eq5_test_mean_tensor = (
            black_scholes_call(
                t=t_test,
                S=S_test,
                K=K,
                T=T,
                r=r_pinn_mean,
                sigma=sigma_pinn_mean
            )
        )

    V_pinn_eq5_test_mean = (
        V_pinn_eq5_test_mean_tensor
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    mse_3_pinn_eq5 = float(
        np.mean(
            (
                V_pinn_eq5_test_mean
                - V_test_np
            ) ** 2
        )
    )

    history_all_runs_n = pd.concat(
        histories_n,
        ignore_index=True
    )

    history_all_runs_n.to_csv(
        (
            f"notebook13_1_random_mixed_"
            f"history_n{n_data}_all_runs.csv"
        ),
        index=False
    )

    history_mean_n = (
        history_all_runs_n
        .drop(columns=["run"])
        .groupby(
            "epoch",
            as_index=False
        )
        .mean()
    )

    history_mean_n.to_csv(
        (
            f"notebook13_1_random_mixed_"
            f"history_n{n_data}_mean.csv"
        ),
        index=False
    )

    pinn_result_mean_n = {
        "n_data":
            n_data,

        # Keep Notebook 13 column names for compatibility.
        # These now contain the mean across the 10 PINN repetitions.
        "r_pinn":
            r_pinn_mean,

        "sigma_pinn":
            sigma_pinn_mean,

        "mse_2_pinn_nn_test":
            mse_2_pinn_nn,

        "mse_3_pinn_eq5_test":
            mse_3_pinn_eq5,

        # Extra diagnostics
        "r_pinn_std":
            r_pinn_std,

        "sigma_pinn_std":
            sigma_pinn_std,

        "mse_2_run_mean":
            float(np.mean(mse2_runs)),

        "mse_2_run_std":
            float(np.std(mse2_runs, ddof=1)),

        "mse_3_run_mean":
            float(np.mean(mse3_runs)),

        "mse_3_run_std":
            float(np.std(mse3_runs, ddof=1)),

        "runtime_minutes":
            float(np.mean(runtime_runs)),

        "runtime_total_minutes":
            float(np.sum(runtime_runs)),

        "pinn_runs":
            PINN_RUNS,
    }

    # Merge unchanged Regression + averaged PINN result
    result_n = {
        **regression_result_n,
        **pinn_result_mean_n,
    }

    results.append(
        result_n
    )

    test_predictions_df[
        f"regression_eq5_n{n_data}"
    ] = V_regression_test_n

    test_predictions_df[
        f"pinn_nn_mean10_n{n_data}"
    ] = V_pinn_nn_test_mean

    test_predictions_df[
        f"pinn_eq5_from_mean_rsigma_n{n_data}"
    ] = V_pinn_eq5_test_mean

    # Save aggregate results after every completed n_data
    pd.DataFrame(
        results
    ).to_csv(
        "notebook13_1_random_mixed_test_mse_results.csv",
        index=False
    )

    pd.DataFrame(
        all_pinn_run_results
    ).to_csv(
        "notebook13_1_random_mixed_all_pinn_runs.csv",
        index=False
    )

    print()
    print("=" * 78)
    print(
        f"AVERAGED PINN RESULT: n_data = {n_data}"
    )
    print("=" * 78)
    print(
        f"mean r_PINN     = {r_pinn_mean:.10f}"
    )
    print(
        f"std  r_PINN     = {r_pinn_std:.10f}"
    )
    print(
        f"mean sigma_PINN = {sigma_pinn_mean:.10f}"
    )
    print(
        f"std  sigma_PINN = {sigma_pinn_std:.10f}"
    )
    print()
    print(
        "MSE #2 from averaged NN prediction =",
        mse_2_pinn_nn
    )
    print(
        "MSE #3 from mean r,sigma -> Eq.(5) =",
        mse_3_pinn_eq5
    )
    print("=" * 78)

# Save all test predictions
test_predictions_df.to_csv(
    "notebook13_1_random_mixed_test_predictions.csv",
    index=False
)

results_df = pd.DataFrame(
    results
)

print()
print(
    "All n_data experiments completed: "
    f"{PINN_RUNS} PINN runs per n_data."
)


In [ ]:
final_columns = [
    "n_data",
    "r_tilde",
    "sigma_tilde",
    "regression_train_mse",
    "r_pinn",
    "sigma_pinn",
    "r_pinn_std",
    "sigma_pinn_std",
    "mse_1_regression_eq5_test",
    "mse_2_pinn_nn_test",
    "mse_3_pinn_eq5_test",
    "mse_2_run_mean",
    "mse_2_run_std",
    "mse_3_run_mean",
    "mse_3_run_std",
    "runtime_minutes",
    "runtime_total_minutes",
    "pinn_runs",
]

final_results = (
    results_df[
        final_columns
    ]
    .copy()
)

final_results


In [ ]:
results_df[
    [
        "n_data",
        "r_tilde",
        "sigma_tilde",
        "r_pinn",
        "sigma_pinn",
        "r_pinn_std",
        "sigma_pinn_std",
        "mse_1_regression_eq5_test",
        "mse_2_pinn_nn_test",
        "mse_3_pinn_eq5_test",
    ]
]


In [ ]:
for n_data in data_sizes:

    history = pd.read_csv(
        (
            f"notebook13_1_random_mixed_"
            f"history_n{n_data}_mean.csv"
        )
    )

    print(
        "\nn_data =", n_data,
        f"(mean history across {PINN_RUNS} runs)"
    )

    print(
        history.iloc[-1][
            [
                "total_loss",
                "data_loss",
                "pde_loss",
                "terminal_loss",
                "boundary_loss",
                "r_estimate",
                "sigma_estimate"
            ]
        ]
    )


In [ ]:
comparison = results_df[
    [
        "n_data",
        "r_tilde",
        "sigma_tilde",
        "r_pinn",
        "sigma_pinn",
        "r_pinn_std",
        "sigma_pinn_std",
        "mse_1_regression_eq5_test",
        "mse_2_pinn_nn_test",
        "mse_3_pinn_eq5_test"
    ]
]

comparison
